# POMCA

> **Contexto de dominio:** [`docs/fuentes/pomca.md`](../../docs/fuentes/pomca.md)  
> **Bloque:** A | **Línea:** `pomca`  
> **Variable principal:** `caudal` (m³/s)  
> **Modelos sugeridos:** SARIMA, PROPHET, XGBOOST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/pomca.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.config import ENSO_LAG_MESES
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "pomca"
VARIABLE = "caudal"
UNIDAD = "m³/s"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `pomca` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "pomca.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/pomca.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/pomca.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "caudal": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

<!-- ENRICHMENT: pomca -->

## 1b. OHDS, IVH e indices POMCA — Balance hidrico de la cuenca

El POMCA articula seis indices hidrologicos para el diagnostico y la zonificacion:

| Indice | Formula simplificada | Umbral critico |
|---|---|---|
| **IUA** | Demanda_total / OHDS | IUA > 50% = alto estres hidrico |
| **IRH** | Caudal_base / Caudal_total | IRH < 0.5 = baja regulacion |
| **IVH** | f(IUA, IRH) | IVH > 3 = vulnerabilidad alta |
| **ICA** | Agregado fisicoquimico | < 0.5 = calidad deficiente |
| **IACAL** | Cargas_vertidas / OHDS | > 2 = alta presion contaminacion |
| **OHDS** | Oferta_total - Caudal_ambiental | -- |

**OHDS (Oferta Hidrica Disponible Superficial):** fraccion del caudal total que puede extraerse respetando el caudal ambiental (Estudio Nacional del Agua, IDEAM).

In [ ]:
# Indices POMCA sinteticos: IUA, IRH, IVH, OHDS por subcuenca
import numpy as np, pandas as pd
np.random.seed(11)
n_sub = 10
subcuencas = [f'SC-{i+1:02d}' for i in range(n_sub)]

oferta_total  = np.random.uniform(10, 200, n_sub)   # hm3/ano
q_ambiental   = oferta_total * np.random.uniform(0.15, 0.30, n_sub)  # 15-30%
ohds          = oferta_total - q_ambiental           # oferta disponible
demanda_total = np.random.uniform(5, 180, n_sub)     # hm3/ano

iua = np.clip(demanda_total / ohds * 100, 0, 200)    # %
irh = np.random.uniform(0.3, 0.85, n_sub)            # indice regulacion

# IVH: vulnerabilidad desabastecimiento (IDEAM: cruza IUA e IRH)
def ivh_score(iua_v, irh_v):
    iua_cat = 3 if iua_v > 50 else 2 if iua_v > 20 else 1
    irh_cat = 3 if irh_v < 0.4 else 2 if irh_v < 0.6 else 1
    return iua_cat + irh_cat  # 2=bajo, 3-4=medio, 5-6=alto

ivh = np.array([ivh_score(iua[i], irh[i]) for i in range(n_sub)])
ivh_cat = pd.cut(ivh, bins=[1,2,4,6], labels=['Bajo','Medio','Alto'])

df_pomca_indices = pd.DataFrame({
    'subcuenca': subcuencas, 'oferta_hm3': oferta_total.round(1),
    'ohds_hm3': ohds.round(1), 'demanda_hm3': demanda_total.round(1),
    'iua_pct': iua.round(1), 'irh': irh.round(3), 'ivh': ivh, 'ivh_cat': ivh_cat
})

print('Indices POMCA por subcuenca:')
print(df_pomca_indices[['subcuenca','ohds_hm3','iua_pct','irh','ivh_cat']])
print(f'\nSubcuencas IUA > 50% (estres hidrico): {(iua > 50).sum()}/{n_sub}')
print(f'Subcuencas IVH Alto (vulnerabilidad desabastecimiento): {(ivh >= 5).sum()}/{n_sub}')

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `pomca` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_pomca.html",
       title="EDA — POMCA", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "caudal", title="POMCA — caudal (m³/s)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "caudal", period="month")
plt.show()

## 3b. Covariable ENSO (ONI)
> Lag recomendado para `pomca` definido en `config.ENSO_LAG_MESES`.

In [ ]:
# --- Covariable ENSO (lag específico para POMCA) ---
# Guard (M-20): avisar si LINEA no tiene lag específico — se aplicará el default.
if LINEA not in ENSO_LAG_MESES:
    _lag_default = ENSO_LAG_MESES["default"]
    print(f"[aviso] '{LINEA}' sin lag específico en ENSO_LAG_MESES; "
          f"se usará default ({_lag_default} meses).")
else:
    print(f"[ok] ENSO lag para '{LINEA}' = {ENSO_LAG_MESES[LINEA]} meses")

oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica=LINEA)
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["caudal"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas
> Compara `caudal` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["caudal"], variable="caudal")
if rep.empty:
    print("Sin normas colombianas registradas para 'caudal'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["caudal"], method="linear")
print(f"Faltantes antes: {df["caudal"].isna().sum()} | "
      f"después: {df_clean["caudal"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["caudal"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: pomca -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Hidrología / cuencas

- **Indicadores ENA:** IUA, IRH, IACAL, ICA agregados a la cuenca/subcuenca.
- **Lag ENSO = 4 meses** (alineado con oferta hídrica).
- **Escalas espaciales:** subcuenca → microcuenca → predio — agregar respetando jerarquía.

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** POMCA (`pomca`)
- **Variable analizada:** `caudal` (m³/s)
- **Modelos ejecutados:** SARIMA, PROPHET, XGBOOST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/pomca.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.